
### **What is Docling?**

**Docling** is an open-source Python library designed to convert complex PDF documents into machine-readable formats like JSON or Markdown. It addresses a long-standing challenge in document processing: extracting structured data from PDFs, which are typically optimized for printing and lack consistent formatting or metadata.


### **Docling Capabilities**

    1. Efficiently converts PDF documents into structured JSON or Markdown formats — fast and highly stable.
    
    2. Accurately interprets page layout, reading order, and table structures, while identifying figures and visual elements.
    
    3. Extracts rich metadata including title, authors, references, and language from documents.
    
    Supports OCR for scanned PDFs, enabling text extraction from image-based content.
    Offers configurable modes:

        1. Batch mode: optimized for high throughput and fast processing.
        2. Interactive mode: tuned for responsiveness with minimal latency.


Compatible with various hardware accelerators like GPU, MPS, and others for enhanced performance.


####  **When to Use Docling in RAG Workflows**

| Scenario | Use Docling? | Why |
|----------|--------------|-----|
| PDF reports or research papers | yes | Preserves structure and enables chunked retrieval |
| Legal or policy documents | yes | Maintains formatting and semantic clarity |
| Web pages or HTML content | yes | Converts to clean Markdown for LLM input |
| Plain text or already structured data | no | Docling may be unnecessary; use basic loaders |



In [1]:
#pip install langchain-docling

In [ ]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType
FILE_PATH = "https://arxiv.org/pdf/2408.09869"

loader = DoclingLoader(file_path=FILE_PATH) # ExportType.CHUNKS  export_type=ExportType.MARKDOWN
docs = loader.load()

In [ ]:
print(docs)

In [18]:
for d in docs[:3]:
    print(f"- {d.page_content=}")

- d.page_content='Version 1.0\nChristoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos 
Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubham Gupta 
Rafael Teixeira de Lima Valery Weber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk Peter W. J. Staar\nAI4K Group, 
IBM Research R¨ uschlikon, Switzerland'

- d.page_content='Abstract\nThis technical report introduces Docling , an easy to use, self-contained, MITlicensed 
open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models for layout
analysis (DocLayNet) and table structure recognition (TableFormer), and runs efficiently on commodity hardware in a
small resource budget. The code interface allows for easy extensibility and addition of new features and models.'

- d.page_content='1 Introduction\nConverting PDF documents back into a machine-processable format has been a major 
challenge for decades due to their huge variability in formats, weak standardization and printing-optimized 
characteristic, which discards most structural features and metadata. With the advent of LLMs and popular 
application patterns such as retrieval-augmented generation (RAG), leveraging the rich content embedded in PDFs has
become ever more relevant. In the past decade, several powerful document understanding solutions have emerged on 
the market, most of which are commercial software, cloud offerings [3] and most recently, multi-modal 
vision-language models. As of today, only a handful of open-source tools cover PDF conversion, leaving a 
significant feature and quality gap to proprietary solutions.\nWith Docling , we open-source a very capable and 
efficient document conversion tool which builds on the powerful, specialized AI models and datasets for layout 
analysis and table structure recognition we developed and presented in the recent past [12, 13, 9]. Docling is 
designed as a simple, self-contained python library with permissive license, running entirely locally on commodity 
hardware. Its code architecture allows for easy extensibility and addition of new features and models.\nHere is 
what Docling delivers today:'

In [15]:
len(docs)

49

In [19]:
def categorize_docs(docs):
    # Initialize three separate lists for different content types
    text_docs, table_docs, image_docs = [], [], []

    # Loop through each LangChain Document returned by DoclingLoader
    for d in docs:
        # Extract Docling-specific metadata (dl_meta) safely
        dl_meta = d.metadata.get("dl_meta", {})
        # Each document chunk has a list of 'doc_items' — layout elements
        doc_items = dl_meta.get("doc_items", [])

        # Collect all labels (e.g., 'text', 'table', 'figure', 'caption') from the items
        labels = {item.get("label", "") for item in doc_items}

        # Categorize based on content labels:
        #  - If it contains a table, treat it as a table chunk
        #  - If it contains a figure or caption, treat it as an image chunk
        #  - Otherwise, treat it as plain text
        if "table" in labels:
            d.metadata["chunk_type"] = "table"
            table_docs.append(d)
        elif "figure" in labels or "caption" in labels:
            d.metadata["chunk_type"] = "image"
            image_docs.append(d)
        else:
            d.metadata["chunk_type"] = "text"
            text_docs.append(d)
    
    # Return the three categorized lists for downstream use (e.g., separate RAG indexes)
    return text_docs, table_docs, image_docs


# Apply categorization to your loaded documents
text_docs, table_docs, image_docs = categorize_docs(docs)


In [20]:
from rich import print 
print(table_docs, image_docs)

[
    Document(
        metadata={
            'source': 'https://arxiv.org/pdf/2408.09869',
            'dl_meta': {
                'schema_name': 'docling_core.transforms.chunker.DocMeta',
                'version': '1.0.0',
                'doc_items': [
                    {
                        'self_ref': '#/texts/69',
                        'parent': {'$ref': '#/body'},
                        'children': [],
                        'content_layer': 'body',
                        'label': 'text',
                        'prov': [
                            {
                                'page_no': 4,
                                'bbox': {
                                    'l': 108.0,
                                    't': 89.39499999999998,
                                    'r': 504.003,
                                    'b': 69.93399999999997,
                                    'coord_origin': 'BOTTOMLEFT'
                                },
                                'charspan': [0, 192]
                            }
                        ]
                    },
                    {
                        'self_ref': '#/texts/71',
                        'parent': {'$ref': '#/body'},
                        'children': [],
                        'content_layer': 'body',
                        'label': 'text',
                        'prov': [
                            {
                                'page_no': 5,
                                'bbox': {
                                    'l': 108.0,
                                    't': 716.523,
                                    'r': 504.003,
                                    'b': 697.062,
                                    'coord_origin': 'BOTTOMLEFT'
                                },
                                'charspan': [0, 121]
                            }
                        ]
                    },
                    {
                        'self_ref': '#/texts/72',
                        'parent': {'$ref': '#/tables/0'},
                        'children': [],
                        'content_layer': 'body',
                        'label': 'caption',
                        'prov': [
                            {
                                'page_no': 5,
                                'bbox': {
                                    'l': 108.0,
                                    't': 685.348,
                                    'r': 504.003,
                                    'b': 644.069,
                                    'coord_origin': 'BOTTOMLEFT'
                                },
                                'charspan': [0, 383]
                            }
                        ]
                    },
                    {
                        'self_ref': '#/tables/0',
                        'parent': {'$ref': '#/body'},
                        'children': [{'$ref': '#/texts/72'}],
                        'content_layer': 'body',
                        'label': 'table',
                        'prov': [
                            {
                                'page_no': 5,
                                'bbox': {
                                    'l': 133.27706909179688,
                                    't': 634.9401092529297,
                                    'r': 478.2610778808594,
                                    'b': 542.4000549316406,
                                    'coord_origin': 'BOTTOMLEFT'
                                },
                                'charspan': [0, 0]
                            }
                        ]
                    }
                ],
                'headings': ['4 Performance'],
                'origin': {
                    'mimetype': 'application/pdf',
                    'binary_hash': 11465328351749295394,
                    'filename': '2408.09869v5.pdf'
                }
            },
            '

In [68]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,
                                               chunk_overlap=50,
                                               add_start_index=True,
                                               separators=["\n","\n\n",""," "]
)
all_splits = text_splitter.split_documents(docs)

len(all_splits)

127

In [22]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

In [23]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

2025-10-26 07:17:58,710 - WARNING - From C:\Users\siddhanna.janai\AppData\Roaming\Python\Python313\site-packages\tf_keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.

2025-10-26 07:18:00,584 - INFO - Use pytorch device_name: cpu
2025-10-26 07:18:00,586 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2


In [63]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
repo_id ="meta-llama/Llama-3.2-3B-Instruct"  #"deepseek-ai/DeepSeek-R1-0528"

llm = HuggingFaceEndpoint(
    repo_id=repo_id,
    max_length=500,
    temperature=0.5,
    provider="auto",  # set your provider here hf.co/settings/inference-providers
    # provider="hyperbolic",
    # provider="nebius",
    # provider="together",
)
llm=ChatHuggingFace(llm=llm)

2025-10-26 07:27:11,516 - WARNING - WARNING! max_length is not default parameter.
                    max_length was transferred to model_kwargs.
                    Please make sure that max_length is what you intended.


In [64]:
llm.invoke("Hi")

AIMessage(content='How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 36, 'total_tokens': 44}, 'model_name': 'meta-llama/Llama-3.2-3B-Instruct', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='run--149d322d-c74e-464d-8325-c7859226dc48-0', usage_metadata={'input_tokens': 36, 'output_tokens': 8, 'total_tokens': 44})

In [50]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [51]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

[
    '2206803b-40f5-44f1-9268-7f2397b3eb7d',
    '8fa7468e-0255-4695-b142-534a43d99adc',
    'cd2ba943-5abf-4ae6-a3f0-6a2359df8ade'
]

In [52]:
vector_store.similarity_search("What are Runtime characteristics of Docling described in table ?",filter={'chunk_type':['table']},k=8)

[Document(id='80717edc-b7ea-4b33-a5af-82962702b607', metadata={'source': 'https://arxiv.org/pdf/2408.09869', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/69', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 4, 'bbox': {'l': 108.0, 't': 89.39499999999998, 'r': 504.003, 'b': 69.93399999999997, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 192]}]}, {'self_ref': '#/texts/71', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 5, 'bbox': {'l': 108.0, 't': 716.523, 'r': 504.003, 'b': 697.062, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 121]}]}, {'self_ref': '#/texts/72', 'parent': {'$ref': '#/tables/0'}, 'children': [], 'content_layer': 'body', 'label': 'caption', 'prov': [{'page_no': 5, 'bbox': {'l': 108.0, 't': 685.348, 'r': 504.003, 'b': 644.069, 'coord_origin': 'BOTTOMLEFT'}, 'charspan':

In [65]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query,k=8,filter={'chunk_type':['table']})
    serialized = "\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [66]:
from langgraph.prebuilt import create_react_agent

tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a blog post. "
    "Use the tool to help answer user queries."
)
agent = create_react_agent(llm, tools, prompt=prompt)

In [67]:
query=" Runtime characteristics of Docling"
for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

 Runtime characteristics of Docling
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (chatcmpl-tool-a0b6f67b34d0477ca6f7854675e5d7b6)
 Call ID: chatcmpl-tool-a0b6f67b34d0477ca6f7854675e5d7b6
  Args:
    query: What is Docling?
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://arxiv.org/pdf/2408.09869', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/69', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 4, 'bbox': {'l': 108.0, 't': 89.39499999999998, 'r': 504.003, 'b': 69.93399999999997, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 192]}]}, {'self_ref': '#/texts/71', 'parent': {'$ref': '#/body'}, 'children': [], 'content_laye